# Fase 5: Evaluación Comparativa
## CNN propia vs Transfer Learning

Comparamos ambos modelos y seleccionamos el mejor para producción.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from src.utils.config import TRAINING_DIR, TESTING_DIR, IMG_SIZE, MODELS_DIR, CLASSES
from src.data.loader import load_image_paths, load_and_preprocess_image
from src.data.preprocessing import encode_labels
from src.evaluation.metrics import evaluate_model
from src.evaluation.plots import plot_confusion_matrix, plot_per_class_metrics

%matplotlib inline

In [ ]:
print("Cargando Testing oficial...")
test_paths, test_labels = load_image_paths(TESTING_DIR)

X_test = []
y_test = []
for path, label in zip(test_paths, test_labels):
    img = load_and_preprocess_image(path, IMG_SIZE)
    if img is not None:
        X_test.append(img)
        y_test.append(label)

X_test = np.array(X_test)
y_test = encode_labels(np.array(y_test))
print(f"Testing: {X_test.shape}")

In [ ]:
import tensorflow as tf

cnn_model = tf.keras.models.load_model(os.path.join(MODELS_DIR, 'cnn_custom.keras'))
resnet_model = tf.keras.models.load_model(os.path.join(MODELS_DIR, 'resnet50_finetuned.keras'))

print("Modelos cargados correctamente.")

In [ ]:
print("=" * 50)
print("EVALUACIÓN - CNN PROPIA")
print("=" * 50)
cnn_global, cnn_per_class, cnn_cm = evaluate_model(cnn_model, X_test, y_test)

print(f"Accuracy:  {cnn_global['accuracy']:.4f}")
print(f"Precision: {cnn_global['precision']:.4f}")
print(f"Recall:    {cnn_global['recall']:.4f}")
print(f"F1-Score:  {cnn_global['f1_score']:.4f}")

print("\n" + "=" * 50)
print("EVALUACIÓN - TRANSFER LEARNING (ResNet50)")
print("=" * 50)
resnet_global, resnet_per_class, resnet_cm = evaluate_model(resnet_model, X_test, y_test)

print(f"Accuracy:  {resnet_global['accuracy']:.4f}")
print(f"Precision: {resnet_global['precision']:.4f}")
print(f"Recall:    {resnet_global['recall']:.4f}")
print(f"F1-Score:  {resnet_global['f1_score']:.4f}")

In [ ]:
print("=" * 60)
print("COMPARATIVA DE MÉTRICAS")
print("=" * 60)
print(f"{'Métrica':<15} {'CNN Propia':<15} {'ResNet50':<15} {'Ganador':<15}")
print("-" * 60)
for metric in ['accuracy', 'precision', 'recall', 'f1_score']:
    cnn_val = cnn_global[metric]
    resnet_val = resnet_global[metric]
    winner = "CNN Propia" if cnn_val >= resnet_val else "ResNet50"
    print(f"{metric:<15} {cnn_val:<15.4f} {resnet_val:<15.4f} {winner:<15}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_confusion_matrix(cnn_cm, '../docs/comparativa_cnn_confusion.png')
plot_confusion_matrix(resnet_cm, '../docs/comparativa_resnet_confusion.png')

### Conclusión

Seleccionamos el modelo con mejor recall (métrica crítica en contexto médico).

**Modelo seleccionado para producción**: Completar según resultados.